# 🏗️ Construction Cost Overrun Analysis
## 建筑工程成本超支驱动因素分析

**Author:** [Your Name]  
**Date:** July 2026  
**Stack:** Python · pandas · matplotlib · numpy

---

## 1. Introduction / 问题背景

Construction projects worldwide suffer from chronic budget overruns. A 2022 McKinsey report found that large projects typically exceed budgets by **20-45%**. In China's booming construction sector, cost control is an everyday battle for project managers.

### Why this analysis?

Most post-mortems blame "unforeseen circumstances" or "market conditions." But what if we could **quantify the specific drivers**? Which factors matter most? How much does each additional change order cost?

### What we'll do

Using 18 real-style Chinese construction project records, we'll:
1. Profile the dataset: how common are overruns?
2. Compare performance by structure type and region
3. Build a correlation matrix to rank cost drivers
4. Analyze the impact of change orders in detail
5. Present actionable recommendations

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import os

# Auto-detect Chinese font or fallback to English
_fonts = [f.name for f in fm.fontManager.ttflist]
_cjk = ['Microsoft YaHei', 'SimHei', 'SimSun', 'KaiTi', 'Noto Sans CJK SC', 'DejaVu Sans']
_font = next((f for f in _cjk if f in _fonts), 'DejaVu Sans')
plt.rcParams['font.family'] = _font
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
plt.style.use('seaborn-v0_8-whitegrid')

print(f'Using font: {_font}')

---
## 2. Data Loading & Cleaning / 数据加载

In [ ]:
df = pd.read_csv('data/construction_projects.csv')

# Parse dates
df['start_date'] = pd.to_datetime(df['开工日期'])
df['end_date'] = pd.to_datetime(df['竣工日期'])

# Engineered features
df['actual_duration'] = (df['end_date'] - df['start_date']).dt.days
df['cost_deviation_wan'] = df['实际成本_万元'] - df['预算_万元']
df['cost_deviation_pct'] = (df['cost_deviation_wan'] / df['预算_万元'] * 100).round(1)
df['cost_per_sqm'] = (df['实际成本_万元'] * 10000 / df['建筑面积_平米']).round(0).astype(int)
df['is_over_budget'] = df['是否超预算'].map({'是': True, '否': False})
df['structure'] = df['结构类型'].map({'框架': 'Frame', '框剪': 'Frame-Shear', '钢结构': 'Steel', '砖混': 'Masonry'})
df['region'] = df['地区'].map({'华东': 'E.China', '华北': 'N.China', '华南': 'S.China', '西南': 'SW'})

print(f'✅ Loaded {len(df)} projects')
print(f'   Columns: {", ".join(df.columns[:10])}...')
df.head(3)

---
## 3. Executive Summary / 总体概况

Before diving in, let's get the big picture.

In [ ]:
over = df['is_over_budget'].sum()
total = len(df)

print('=' * 55)
print(f'  📊 {total} projects analyzed')
print(f'  💰 Avg Budget:    {df["预算_万元"].mean():,.0f} wan CNY')
print(f'  💸 Avg Actual:    {df["实际成本_万元"].mean():,.0f} wan CNY')
print(f'  📉 Mean Deviation: {df["cost_deviation_pct"].mean():.1f}%')
print(f'  ⚠️  Over-budget:    {over}/{total} ({over/total*100:.1f}%)')
print('=' * 55)

# Quick distribution
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if v else '#27ae60' for v in df['is_over_budget']]
ax.barh(df['项目名称'], df['cost_deviation_pct'], color=colors, alpha=0.8)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('Cost Deviation by Project (% over/under budget)', fontweight='bold')
ax.set_xlabel('Deviation % (red = over budget)')
plt.tight_layout()
plt.show()

**Takeaway:** Over half of projects went over budget. But the deviation range is wide — from -2.5% (under) to +11.3% (severe overrun). Why?

---
## 4. Structure Type Comparison / 结构类型对比

Different structure types have inherently different cost profiles. Let's compare.

In [ ]:
struct = df.groupby('structure').agg(
    Count=('项目名称', 'count'),
    Avg_Deviation_pct=('cost_deviation_pct', 'mean'),
    Avg_Cost_per_sqm=('cost_per_sqm', 'mean')
).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: deviation
colors1 = ['#e74c3c' if x > 0 else '#27ae60' for x in struct['Avg_Deviation_pct']]
ax1.bar(struct.index, struct['Avg_Deviation_pct'], color=colors1, edgecolor='white')
ax1.axhline(y=0, color='gray', linestyle='--')
ax1.set_title('Avg Cost Deviation by Structure Type', fontweight='bold')
ax1.set_ylabel('Deviation (%)')
for i, v in enumerate(struct['Avg_Deviation_pct']):
    ax1.text(i, v + 0.3 if v >= 0 else v - 1.5, f'{v:+.1f}%', ha='center', fontweight='bold')

# Right: cost per sqm
ax2.bar(struct.index, struct['Avg_Cost_per_sqm'], color=['#f39c12', '#e67e22', '#2ecc71', '#d35400'], edgecolor='white')
ax2.set_title('Avg Cost per m² by Structure Type', fontweight='bold')
ax2.set_ylabel('CNY / m²')
for i, v in enumerate(struct['Avg_Cost_per_sqm']):
    ax2.text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(struct.to_string())

**Insight:** Frame-Shear wall structures (框剪) have the highest deviation (+6.9%) **and** the highest unit cost. Steel structures, by contrast, are actually *under* budget on average — likely because they're prefabricated and more predictable.

---
## 5. Regional Analysis / 地区差异

Weather, labor costs, and local regulations vary by region. China's eastern seaboard is very different from the southwest interior.

In [ ]:
region = df.groupby('region').agg(
    Projects=('项目名称', 'count'),
    Avg_Deviation_pct=('cost_deviation_pct', 'mean'),
    Over_budget_ratio=('is_over_budget', 'mean'),
    Avg_Weather_Delay=('天气延误_天', 'mean')
).round(1)
region['Over_budget_ratio'] = (region['Over_budget_ratio'] * 100).round(0).astype(int)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(region))
w = 0.35
ax.bar(x - w/2, region['Avg_Deviation_pct'], w, label='Avg Deviation (%)', color='#3498db', edgecolor='white')
ax2 = ax.twinx()
ax2.bar(x + w/2, region['Over_budget_ratio'], w, label='Over-budget (%)', color='#e74c3c', alpha=0.7, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(region.index)
ax.set_title('Cost Performance by Region', fontweight='bold')
ax.set_ylabel('Avg Deviation (%)')
ax2.set_ylabel('Over-budget Ratio (%)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

print(region.to_string())

**Insight:** East China has the highest over-budget rate (70%) and most weather delays (21.4 days on average). This makes intuitive sense — coastal typhoons + dense urban construction = more disruption. North China is the most stable region.

---
## 6. Correlation Matrix: What Drives Cost Overruns? / 相关性分析

This is the core of the analysis. Let's compute the Pearson correlation between cost deviation and six potential drivers.

In [ ]:
driver_cols = {
    '变更单数': 'Change Orders',
    '天气延误_天': 'Weather Delay',
    '安全事故数': 'Safety Incidents',
    '层数': 'Floors',
    '建筑面积_平米': 'Floor Area',
    'actual_duration': 'Duration'
}

corr_df = df[list(driver_cols.keys()) + ['cost_deviation_wan']].corr()
cost_corr = corr_df['cost_deviation_wan'].drop('cost_deviation_wan').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
labels = [driver_cols.get(c, c) for c in cost_corr.index]
colors = ['#c0392b' if v > 0.5 else '#e74c3c' if v > 0 else '#27ae60' for v in cost_corr.values]
bars = ax.barh(labels, cost_corr.values, color=colors, edgecolor='white')
ax.set_title('What Drives Cost Overruns?\nPearson Correlation with Cost Deviation', fontweight='bold', fontsize=14)
ax.set_xlabel('Pearson r')
ax.set_xlim(-1, 1)
for bar, val in zip(bars, cost_corr.values):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'r = {val:.3f}', va='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

for col, val in cost_corr.items():
    bar = '█' * int(abs(val) * 15)
    print(f'  {driver_cols.get(col, col):<18s} r = {val:+.3f}  {bar}')

**This is the headline finding.** 

- **Weather delay** shows the strongest linear correlation at r = +0.891. This is partly because weather delays extend duration, and extended duration = more overhead.
- **Change orders** at r = +0.757 are arguably the more *actionable* insight — you can't control the weather, but you *can* control change order management.
- Duration, safety incidents, and number of floors all hover around r = 0.75-0.80, suggesting a cluster of correlated project-complexity factors.

---
## 7. Deep Dive: Change Order Impact / 变更单深度分析

If change orders are the #1 controllable driver, let's quantify the effect.

In [ ]:
df['co_level'] = pd.cut(df['变更单数'], bins=[0, 5, 12, 30], labels=['Low (0-5)', 'Medium (6-12)', 'High (13+)'])
co_stats = df.groupby('co_level', observed=False).agg(
    Projects=('项目名称', 'count'),
    Avg_Deviation_pct=('cost_deviation_pct', 'mean'),
    Over_budget_ratio=('is_over_budget', 'mean')
).round(1)
co_stats['Over_budget_ratio'] = (co_stats['Over_budget_ratio'] * 100).round(0).astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: scatter with trend
colors_scatter = ['#e74c3c' if v else '#27ae60' for v in df['is_over_budget']]
ax1.scatter(df['变更单数'], df['cost_deviation_pct'], c=colors_scatter,
           s=df['建筑面积_平米']/200, alpha=0.7, edgecolors='black', linewidth=0.5)
z = np.polyfit(df['变更单数'], df['cost_deviation_pct'], 1)
p = np.poly1d(z)
xl = np.linspace(df['变更单数'].min(), df['变更单数'].max(), 100)
ax1.plot(xl, p(xl), '--', color='gray', alpha=0.7, linewidth=2)
ax1.set_title('Change Orders → Cost Deviation', fontweight='bold')
ax1.set_xlabel('Number of Change Orders')
ax1.set_ylabel('Cost Deviation (%)')
ax1.annotate(f'Trend line\n(y = {z[0]:.2f}x + {z[1]:.1f})',
            xy=(20, p(20)), xytext=(22, p(20)+2),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

# Right: bar chart of tiers
bar_colors = ['#27ae60', '#f39c12', '#e74c3c']
ax2.bar(co_stats.index, co_stats['Avg_Deviation_pct'], color=bar_colors, edgecolor='white')
ax2.set_title('Avg Deviation by Change Order Level', fontweight='bold')
ax2.set_ylabel('Avg Deviation (%)')
for i, (_, row) in enumerate(co_stats.iterrows()):
    ax2.text(i, row['Avg_Deviation_pct'] + 0.3 if row['Avg_Deviation_pct'] >= 0 else row['Avg_Deviation_pct'] - 0.8,
            f"{row['Avg_Deviation_pct']:+.1f}%\n({row['Over_budget_ratio']}% over)",
            ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print(co_stats.to_string())
print(f'\n📈 Slope: each additional change order correlates with ~{z[0]:.2f} percentage points of deviation')

**This chart tells a clear story:**

- **0-5 change orders:** 0% of projects went over budget. Avg deviation: **-2.5%** (under budget!)
- **6-12 change orders:** 60% over budget, avg +1.5%
- **13+ change orders:** 90% over budget, avg +5.4%

The relationship is nearly stepwise. If you can keep change orders below 6, your project has essentially zero risk of overrun.

---
## 8. Top 5 Problem Cases / 严重超支项目

Who went over the most, and why?

In [ ]:
top5 = df.nlargest(5, 'cost_deviation_pct')

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(top5))
w = 0.25
ax.bar(x - w, top5['cost_deviation_pct'], w, label='Deviation %', color='#c0392b', edgecolor='white')
ax.bar(x, top5['变更单数'], w, label='Change Orders', color='#8e44ad', edgecolor='white')
ax.bar(x + w, top5['天气延误_天'], w, label='Weather Delay (days)', color='#2980b9', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(top5['项目名称'], rotation=15, ha='right', fontsize=8)
ax.set_title('Top 5 Over-budget Projects: Multi-factor Breakdown', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Table
top5[['项目名称', 'cost_deviation_pct', '变更单数', '天气延误_天', '安全事故数', 'structure']].style \
    .background_gradient(subset=['cost_deviation_pct'], cmap='Reds') \
    .format({'cost_deviation_pct': '{:.1f}%'})

**Pattern:** The top 5 all have high change orders AND significant weather delays AND accidents. The worst offender (Binjiang II) had 25 change orders + 42 days of weather delay + 4 safety incidents. This is the "triple-whammy" scenario.

---
## 9. Conclusions & Recommendations / 结论与建议

### 🔑 Three key takeaways

1. **Change orders are the #1 *controllable* cost driver.** With r = +0.757 and a near-stepwise pattern, this is where project managers should focus.

2. **Weather ≠ excuse.** Yes, weather delays have the highest correlation (r = +0.891). But this is partly because delay → more overhead → more cost. Better scheduling and risk reserves can mitigate.

3. **Frame-Shear structures need extra attention.** They average +6.9% deviation. If you're building one, budget a larger contingency fund.

### 💡 Actionable recommendations

| Recommendation | Target Audience | Expected Impact |
|----------------|-----------------|-----------------|
| Tighten change order approval — set a max of 5 per project | PMO / Project Managers | Reduce overrun rate from 55% to <20% |
| Frame-Shear projects: increase risk reserve from 5% to 10% | Cost Estimators | Absorb typical +6.9% deviation |
| East China / Southwest: add weather contingency buffer | Regional Directors | Cover avg 14-21 day weather delays |
| Adopt BIM for change-order impact simulation before approval | Digital Construction Team | Prevent high-cost changes |

### 🔮 What's next?

- **Phase 2:** Get a larger dataset (100+ projects) and build a logistic regression model to predict `P(over_budget | characteristics)`
- **Phase 3:** Streamlit app where PMs input project parameters and get a risk score
- **Phase 4:** Integrate with real-time BIM data for live cost forecasting

---

## About This Project

Built by an Engineering Management student exploring the intersection of construction and data science.  
🇨🇳 东北林业大学 · 工程管理 · Class of 2026

**Skills demonstrated:** Python · pandas · matplotlib · statistical analysis · construction domain knowledge  
**For:** Graduate school applications in CS / Data Science / Construction Informatics

---

*If you found this useful, give it a ⭐ on [GitHub](https://github.com)!*